# Abdallah - BM25 Retrieval Evaluation

**Assigned part:** Evaluate retrieval using MAP@10 and MRR@10.

This notebook evaluates Jomana's BM25 lexical retriever using curated relevance judgments for sample technical questions.

## Scope

Covered here:

- Load the StackLite corpus and BM25 retriever.
- Load curated evaluation questions and relevant document IDs.
- Retrieve top-10 BM25 results for each question.
- Compute MAP@10 and MRR@10.
- Save per-query metrics, top-10 run results, and a short markdown report.

Not covered in this milestone: MiniLM semantic search, hybrid fusion, RAG generation, citation quality review, or UI integration.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/content')]
PROJECT_ROOT = next((path for path in PROJECT_CANDIDATES if (path / 'src').exists()), Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bm25_retriever import BM25Retriever, load_stacklite_zip, write_results_csv
from src.evaluation import evaluate_retrieval, load_evaluation_questions, write_dict_csv

DATASET_PATH = PROJECT_ROOT / 'DataSet.zip'
QUESTIONS_PATH = PROJECT_ROOT / 'evaluation' / 'bm25_eval_queries.json'
PER_QUERY_OUTPUT = PROJECT_ROOT / 'results' / 'bm25_evaluation_per_query.csv'
RUN_OUTPUT = PROJECT_ROOT / 'results' / 'bm25_evaluation_top10.csv'
REPORT_OUTPUT = PROJECT_ROOT / 'reports' / 'bm25_evaluation_report.md'

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset path: {DATASET_PATH}')
print(f'Evaluation questions: {QUESTIONS_PATH}')

Project root: C:\Users\RTX\Downloads\Advanced DL
Dataset path: C:\Users\RTX\Downloads\Advanced DL\DataSet.zip
Evaluation questions: C:\Users\RTX\Downloads\Advanced DL\evaluation\bm25_eval_queries.json


## 1. Load Evaluation Questions

Each question has one or more manually selected relevant StackLite document IDs. These relevance judgments are used to calculate MAP@10 and MRR@10.

In [2]:
questions = load_evaluation_questions(QUESTIONS_PATH)

questions_df = pd.DataFrame([
    {
        'query_id': question.query_id,
        'query': question.query,
        'relevant_doc_ids': ', '.join(sorted(question.relevant_doc_ids)),
        'notes': question.notes,
    }
    for question in questions
])
questions_df

,query_id,query,relevant_doc_ids,notes
0,q1_micro_macro_average,Why are micro average precision recall and F1 ...,datascience:15989,Exact StackLite question about micro and macro...
1,q2_ai_vs_ml,What is the difference between artificial inte...,"ai:35, datascience:19077",Both AI and Data Science Stack Exchange contai...
2,q3_deconvolution_layers,What are deconvolutional layers in convolution...,datascience:6107,Directly relevant question about deconvolution...
3,q4_keras_class_weights,How do I set class weights for imbalanced clas...,datascience:13490,Directly relevant Keras class-weight question.
4,q5_dying_relu,What is the dying ReLU problem in neural netwo...,"datascience:18810, datascience:5706",Includes the direct dying-ReLU question and a ...


## 2. Build BM25 Baseline

The evaluation uses the same BM25 retriever from Jomana's milestone so the reported metrics reflect the actual lexical baseline.

In [3]:
documents = load_stacklite_zip(DATASET_PATH)
bm25 = BM25Retriever(k1=1.5, b=0.75).fit(documents)

print(f'Indexed documents: {len(documents):,}')
print(f'Vocabulary size: {len(bm25.idf):,}')
print(f'Average document length: {bm25.avg_doc_length:.2f} tokens')

Indexed documents: 1,500
Vocabulary size: 13,242
Average document length: 84.92 tokens


## 3. Retrieve Top-10 Results

The top-10 BM25 results are saved for inspection and for later comparison with semantic or hybrid retrieval.

In [4]:
query_to_results = {
    question.query: bm25.search(question.query, top_k=10)
    for question in questions
}

run = {
    question.query_id: [
        str(result['doc_id']) for result in query_to_results[question.query]
    ]
    for question in questions
}

for question in questions:
    print(f"\nQUERY [{question.query_id}]: {question.query}")
    display(pd.DataFrame(query_to_results[question.query])[[
        'rank', 'score', 'doc_id', 'source', 'title', 'link'
    ]])


QUERY [q1_micro_macro_average]: Why are micro average precision recall and F1 equal in multiclass classification?

QUERY [q2_ai_vs_ml]: What is the difference between artificial intelligence and machine learning?

QUERY [q3_deconvolution_layers]: What are deconvolutional layers in convolutional neural networks?

QUERY [q4_keras_class_weights]: How do I set class weights for imbalanced classes in Keras?

QUERY [q5_dying_relu]: What is the dying ReLU problem in neural networks?


,rank,score,doc_id,source,title,link
0,1,48.493982,datascience:15989,datascience,Micro Average vs Macro average Performance in ...,https://datascience.stackexchange.com/question...
1,2,30.747687,datascience:36817,datascience,Why is the F-measure preferred for classificat...,https://datascience.stackexchange.com/question...
2,3,24.953232,datascience:40900,datascience,What's the difference between Sklearn F1 score...,https://datascience.stackexchange.com/question...
3,4,24.292095,datascience:25119,datascience,How to calculate mAP for detection task for th...,https://datascience.stackexchange.com/question...
4,5,22.189749,datascience:45165,datascience,"How to get accuracy, F1, precision and recall,...",https://datascience.stackexchange.com/question...
5,6,19.732273,datascience:64441,datascience,How to interpret classification report of scik...,https://datascience.stackexchange.com/question...
6,7,19.564670,datascience:65839,datascience,macro average and weighted average meaning in ...,https://datascience.stackexchange.com/question...
7,8,18.118757,datascience:30881,datascience,When is precision more important over recall?,https://datascience.stackexchange.com/question...
8,9,15.495841,datascience:16342,datascience,Unbalanced multiclass data with XGBoost,https://datascience.stackexchange.com/question...
9,10,14.826174,datascience:16232,datascience,In XGBoost would we evaluate results with a Pr...,https://datascience.stackexchange.com/question...


,rank,score,doc_id,source,title,link
0,1,22.033671,datascience:19077,datascience,Difference between machine learning and artifi...,https://datascience.stackexchange.com/question...
1,2,18.972563,ai:35,ai,What is the difference between artificial inte...,https://ai.stackexchange.com/questions/35/what...
2,3,16.138720,ai:2738,ai,How is Bayes' Theorem used in artificial intel...,https://ai.stackexchange.com/questions/2738/ho...
3,4,15.919972,ai:2306,ai,What are the top artificial intelligence journ...,https://ai.stackexchange.com/questions/2306/wh...
4,5,15.676151,ai:1462,ai,What is the difference between artificial inte...,https://ai.stackexchange.com/questions/1462/wh...
5,6,15.113489,ai:1847,ai,What is the difference between artificial inte...,https://ai.stackexchange.com/questions/1847/wh...
6,7,14.658173,ai:3175,ai,What are examples of techniques to prevent bia...,https://ai.stackexchange.com/questions/3175/wh...
7,8,14.165380,ai:16646,ai,Is artificial intelligence really just human i...,https://ai.stackexchange.com/questions/16646/i...
8,9,13.934380,ai:7446,ai,What is the difference between artificial inte...,https://ai.stackexchange.com/questions/7446/wh...
9,10,13.650550,ai:16,ai,"What is ""early stopping"" in machine learning?",https://ai.stackexchange.com/questions/16/what...


,rank,score,doc_id,source,title,link
0,1,22.192062,datascience:6107,datascience,What are deconvolutional layers?,https://datascience.stackexchange.com/question...
1,2,17.607478,datascience:15903,datascience,Why do convolutional neural networks work?,https://datascience.stackexchange.com/question...
2,3,13.970970,ai:1978,ai,Do deep learning algorithms represent ensemble...,https://ai.stackexchange.com/questions/1978/do...
3,4,12.967898,ai:21810,ai,What is a fully convolution network?,https://ai.stackexchange.com/questions/21810/w...
4,5,12.577344,ai:13692,ai,When should I use 3D convolutions?,https://ai.stackexchange.com/questions/13692/w...
5,6,12.541757,datascience:51470,datascience,What are the differences between Convolutional...,https://datascience.stackexchange.com/question...
6,7,12.071487,ai:5862,ai,Why aren't there neural networks that connect ...,https://ai.stackexchange.com/questions/5862/wh...
7,8,12.019565,ai:8323,ai,How to handle rectangular images in convolutio...,https://ai.stackexchange.com/questions/8323/ho...
8,9,11.856026,datascience:26597,datascience,How to set the number of neurons and layers in...,https://datascience.stackexchange.com/question...
9,10,11.710486,ai:7865,ai,Which layer in a CNN consumes more training ti...,https://ai.stackexchange.com/questions/7865/wh...


,rank,score,doc_id,source,title,link
0,1,32.996294,datascience:13490,datascience,How to set class weights for imbalanced classe...,https://datascience.stackexchange.com/question...
1,2,23.280822,datascience:44883,datascience,Deep network not able to learn imbalanced data...,https://datascience.stackexchange.com/question...
2,3,18.837304,datascience:36862,datascience,Macro- or micro-average for imbalanced class p...,https://datascience.stackexchange.com/question...
3,4,17.742770,datascience:48369,datascience,What loss function to use for imbalanced class...,https://datascience.stackexchange.com/question...
4,5,14.879306,datascience:44755,datascience,Why doesn't class weight resolve the imbalance...,https://datascience.stackexchange.com/question...
5,6,14.612408,ai:4889,ai,"How to implement an ""unknown"" class in multi-c...",https://ai.stackexchange.com/questions/4889/ho...
6,7,14.427566,datascience:14415,datascience,How does Keras calculate accuracy?,https://datascience.stackexchange.com/question...
7,8,13.442460,datascience:1107,datascience,Quick guide into training highly imbalanced da...,https://datascience.stackexchange.com/question...
8,9,13.247156,datascience:310,datascience,One-Class discriminatory classification with i...,https://datascience.stackexchange.com/question...
9,10,13.134933,datascience:16342,datascience,Unbalanced multiclass data with XGBoost,https://datascience.stackexchange.com/question...


,rank,score,doc_id,source,title,link
0,1,24.127748,datascience:5706,datascience,"What is the ""dying ReLU"" problem in neural net...",https://datascience.stackexchange.com/question...
1,2,21.053690,datascience:18810,datascience,How to check for dead relu neurons,https://datascience.stackexchange.com/question...
2,3,10.786410,ai:9828,ai,What happens when I mix activation functions?,https://ai.stackexchange.com/questions/9828/wh...
3,4,9.889884,datascience:14349,datascience,Difference of Activation Functions in Neural N...,https://datascience.stackexchange.com/question...
4,5,9.754422,ai:2874,ai,Can neural networks efficiently solve the trav...,https://ai.stackexchange.com/questions/2874/ca...
5,6,9.706955,datascience:28328,datascience,How does multicollinearity affect neural netwo...,https://datascience.stackexchange.com/question...
6,7,9.369132,ai:5322,ai,Can prior knowledge be encoded in deep neural ...,https://ai.stackexchange.com/questions/5322/ca...
7,8,9.277289,ai:10944,ai,Are there neural networks with very few nodes ...,https://ai.stackexchange.com/questions/10944/a...
8,9,8.824567,datascience:19272,datascience,Deep Neural Network - Backpropogation with ReLU,https://datascience.stackexchange.com/question...
9,10,8.695480,ai:6892,ai,What tools are used to deal with adversarial e...,https://ai.stackexchange.com/questions/6892/wh...


## 4. Compute MAP@10 and MRR@10

- **AP@10** measures ranking quality for each query across the top 10 results.
- **MAP@10** is the mean AP@10 across all evaluation queries.
- **RR@10** measures how early the first relevant result appears.
- **MRR@10** is the mean RR@10 across all evaluation queries.

In [5]:
metrics, per_query_rows = evaluate_retrieval(run, questions, k=10)

metrics_df = pd.DataFrame([metrics])
per_query_df = pd.DataFrame(per_query_rows)

display(metrics_df)
display(per_query_df[['query_id', 'AP@10', 'RR@10', 'relevant_doc_ids', 'retrieved_doc_ids']])

,MAP@10,MRR@10,queries
0,1.0,1.0,5.0


,query_id,AP@10,RR@10,relevant_doc_ids,retrieved_doc_ids
0,q1_micro_macro_average,1.0,1.0,datascience:15989,"datascience:15989, datascience:36817, datascie..."
1,q2_ai_vs_ml,1.0,1.0,"ai:35, datascience:19077","datascience:19077, ai:35, ai:2738, ai:2306, ai..."
2,q3_deconvolution_layers,1.0,1.0,datascience:6107,"datascience:6107, datascience:15903, ai:1978, ..."
3,q4_keras_class_weights,1.0,1.0,datascience:13490,"datascience:13490, datascience:44883, datascie..."
4,q5_dying_relu,1.0,1.0,"datascience:18810, datascience:5706","datascience:5706, datascience:18810, ai:9828, ..."


## 5. Save Evaluation Outputs

The generated files are useful for the final report and for later comparison against semantic and hybrid search.

In [6]:
write_results_csv(query_to_results, RUN_OUTPUT)
write_dict_csv(per_query_rows, PER_QUERY_OUTPUT)

report_lines = [
    '# BM25 Retrieval Evaluation',
    '',
    "This report covers Abdallah's retrieval evaluation milestone for the BM25 baseline.",
    '',
    '## Metrics',
    '',
    f"- Queries: {int(metrics['queries'])}",
    f"- MAP@10: {metrics['MAP@10']:.6f}",
    f"- MRR@10: {metrics['MRR@10']:.6f}",
    '',
    '## Per-Query Results',
    '',
    '| Query ID | AP@10 | RR@10 | Relevant Documents |',
    '|---|---:|---:|---|',
]
for row in per_query_rows:
    report_lines.append(
        f"| {row['query_id']} | {row['AP@10']:.6f} | {row['RR@10']:.6f} | {row['relevant_doc_ids']} |"
    )
report_lines.extend([
    '',
    '## Notes',
    '',
    '- MAP@10 measures how highly relevant documents are ranked across the top 10 results.',
    '- MRR@10 measures how early the first relevant document appears.',
    '- Relevance judgments are stored in `evaluation/bm25_eval_queries.json`.',
    '- This evaluation is limited to BM25 retrieval and does not include semantic search or RAG generation.',
    '',
])
REPORT_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
REPORT_OUTPUT.write_text('\n'.join(report_lines), encoding='utf-8')

print(f'Saved per-query metrics: {PER_QUERY_OUTPUT}')
print(f'Saved top-10 run: {RUN_OUTPUT}')
print(f'Saved report: {REPORT_OUTPUT}')

Saved per-query metrics: C:\Users\RTX\Downloads\Advanced DL\results\bm25_evaluation_per_query.csv
Saved top-10 run: C:\Users\RTX\Downloads\Advanced DL\results\bm25_evaluation_top10.csv
Saved report: C:\Users\RTX\Downloads\Advanced DL\reports\bm25_evaluation_report.md


## Result Summary

The BM25 baseline performs strongly on these curated exact-match technical questions. This is expected because BM25 is very effective when query wording overlaps with source titles and tags. The limitation is that synonym-heavy or paraphrased questions may require semantic retrieval, which belongs to the next milestone.